# VRP Toolkit Quick Start

Welcome to the VRP Toolkit! This tutorial will guide you through solving a Pickup and Delivery Problem with Time Windows (PDPTW) using Adaptive Large Neighborhood Search (ALNS).

## What You'll Learn
- Generate synthetic map data for testing
- Create demand data for restaurant-customer pairs
- Build a PDPTW problem instance
- Generate an initial solution using greedy insertion
- Improve the solution using ALNS algorithm
- Visualize the solution

**Time estimate:** 3-5 minutes

## 1. Import Required Modules

First, let's import the necessary modules from the VRP Toolkit:

In [ ]:
import numpy as np
import pandas as pd
import random

# Import VRP Toolkit modules
from vrp_toolkit.data.map import RealMap
from vrp_toolkit.data.generators import DemandGenerator, OrderGenerator
from vrp_toolkit.problems.pdptw import PDPTWInstance
from vrp_toolkit.algorithms.alns.solver import greedy_insertion_initial_solution, ALNS, ALNSConfig

## 2. Setup Reproducibility

Set random seeds to ensure reproducible results:

In [ ]:
# Set random seeds for reproducibility
seed_value = 42
np.random.seed(seed_value)
random.seed(seed_value)

## 3. Create a Synthetic Map

We'll start by creating a synthetic map with random coordinates. This simulates a small delivery area with:
- 2 restaurants (pickup locations)
- 4 customers (delivery locations)
- Random Euclidean distances between locations

In [ ]:
# Create a synthetic map
realMap = RealMap(
    n_r=2,  # number of restaurants
    n_c=4,  # number of customers
    dist_function=np.random.uniform,
    dist_params={'low': -1, 'high': 1}
)

print(f"Map created with {realMap.N_R} restaurants and {realMap.N_C} customers")
print(f"Total nodes: {len(realMap.all_nodes)}")

## 4. Generate Demand Data

Next, we generate demand data for restaurant-customer pairs across time intervals. Each pair has:
- Random number of orders (Poisson distribution)
- Time windows spanning 30 minutes with 10-minute intervals

In [ ]:
# Define demand generation parameters
random_params = {
    'sample_dist': {'function': np.random.randint, 'params': {'low': 1, 'high': 3}},
    'demand_dist': {'function': np.random.poisson, 'params': {'lam': 2}}
}

# Generate demands
demands = DemandGenerator(
    time_range=30,
    time_step=10,
    restaurants=realMap.restaurants,
    customers=realMap.customers,
    random_params=random_params
)

print(f"Demand table shape: {demands.demand_table.shape}")
print(f"Number of time intervals: {demands.time_intervals}")

## 5. Create PDPTW Orders

Now we convert demand data into PDPTW orders. Each order includes:
- Pickup node (restaurant)
- Delivery node (customer)
- Time window constraints
- Service times
- Robot speed (4 mph)

In [ ]:
# Time parameters for PDPTW
time_params = {
    'time_window_length': 30,
    'service_time': 5,
    'extra_time': 10
}

# Create PDPTW orders
pdptw_order = OrderGenerator(
    realMap,
    demands.demand_table,
    time_params,
    robot_speed=4
)

print(f"Total number of orders: {pdptw_order.total_number_orders}")

# Display first few rows of the order table
pd.set_option('display.max_columns', None)
pdptw_order.order_table.head()

## 6. Create PDPTW Instance

Convert the order table into a formal PDPTW instance that includes:
- Distance matrix
- Time windows
- Node information
- Battery and capacity constraints

In [ ]:
# Create PDPTW instance
pdptw_instance = PDPTWInstance(pdptw_order)

print(f"Instance created with {pdptw_instance.n} orders")
print(f"Distance matrix shape: {pdptw_instance.distance_matrix.shape}")
print(f"Robot speed: {pdptw_instance.robot_speed} mph")

## 7. Generate Initial Solution

We'll use greedy insertion to create an initial feasible solution. This solution:
- Assigns orders to 4 vehicles
- Respects vehicle capacity (6 orders per vehicle)
- Considers battery constraints
- Includes penalties for unvisited or delayed orders

In [ ]:
# Helper function for battery capacity calculation
def battery_relaxation(battery, dist_matrix, robot_speed, indicator=None):
    if indicator:
        battery_capacity = (battery - np.mean(dist_matrix[0][1:-1])) * 2 / robot_speed * 60
    else:
        battery_capacity = battery / robot_speed * 60
    return battery_capacity

# Algorithm parameters
num_vehicles = 4
vehicle_capacity = 6
battery_consume_rate = 1
penalty_unvisited = 100
penalty_delayed = 15

# Battery settings
battery = 8  # miles
dist_matrix = pdptw_instance.distance_matrix
robot_speed = pdptw_instance.robot_speed
if_battery_relaxation = 1
battery_capacity = battery_relaxation(battery, dist_matrix, robot_speed, if_battery_relaxation)

# Generate initial solution using greedy insertion
initial_solution = greedy_insertion_initial_solution(
    pdptw_instance,
    num_vehicles,
    vehicle_capacity,
    battery_capacity,
    battery_consume_rate,
    penalty_unvisited,
    penalty_delayed
)

print(f"Initial solution objective value: {initial_solution.objective_function():.2f}")
print(f"Number of routes: {len(initial_solution.routes)}")

## 8. Configure ALNS Algorithm

Now we'll set up the ALNS algorithm to improve the initial solution. ALNS uses:
- Removal operators (Shaw, Random, Worst, SISR)
- Repair operators (Greedy, Regret)
- Simulated annealing for acceptance
- Adaptive operator selection

In [ ]:
# Helper function for distance-time matrix
def generate_d_matrix(instance):
    n = instance.n
    robot_speed = instance.robot_speed
    dist_matrix = instance.distance_matrix
    start_time = np.array([instance.time_windows[i][0] for i in range(1, n+1)])
    end_time = np.array([instance.time_windows[i][1] for i in range(1, n+1)])
    
    d_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            # pickup-pickup
            d_1 = dist_matrix[i+1][j+1]
            t_1 = abs(start_time[i] - start_time[j]) / 60 * robot_speed 
            dt_1 = d_1 + t_1 * 0.3

            # pickup-dropoff
            d_2 = dist_matrix[i+1][j+n+1]
            t_2 = abs(start_time[i] - end_time[j]) / 60 * robot_speed 
            dt_2 = d_2 + t_2 * 0.3

            # dropoff-pickup
            d_3 = dist_matrix[i+n+1][j+1]
            t_3 = abs(start_time[j] - end_time[i]) / 60 * robot_speed 
            dt_3 = d_3 + t_3 * 0.3

            # dropoff-dropoff
            d_4 = dist_matrix[i+n+1][j+n+1]
            t_4 = abs(start_time[j] - end_time[i]) / 60 * robot_speed 
            dt_4 = d_4 + t_4 * 0.3

            d_matrix[i][j] = min(dt_1, dt_2, dt_3, dt_4)

    return d_matrix

# Generate distance-time matrix
d_matrix = generate_d_matrix(pdptw_instance)

# Operator parameters
params_operators = {
    'num_removal': int(pdptw_instance.n * 0.3),
    'p': 3,
    'k': 3,
    'L_max': 6,
    'avg_remove_order': 6,
    'd_matrix': d_matrix
}

# ALNS algorithm parameters
max_no_improve = 25
segment_length = 10
num_segments = 15
r = 0.2  # weight update rate
sigma = [10, 5, 1]  # acceptance scores
start_temp = 100
cooling_rate = 0.99

# Create ALNS solver
config = ALNSConfig(
    num_removal=params_operators['num_removal'],
    p=params_operators['p'],
    k=params_operators['k'],
    L_max=params_operators['L_max'],
    avg_remove_order=params_operators['avg_remove_order'],
    d_matrix=params_operators['d_matrix'],
    max_no_improve=max_no_improve,
    segment_length=segment_length,
    num_segments=num_segments,
    r=r,
    sigma=tuple(sigma),  # Convert list to tuple
    start_temp=start_temp,
    cooling_rate=cooling_rate
)

alns = ALNS(
    initial_solution=initial_solution,
    config=config,
    dist_matrix=dist_matrix,
    battery_capacity=battery_capacity
)

print("ALNS solver created successfully")

## 9. Run ALNS Algorithm

Execute the ALNS algorithm to improve the solution. The algorithm will:
- Run for 15 segments
- Apply removal and repair operators
- Adaptively select the best operators
- Use simulated annealing for solution acceptance
- Track progress and improvements

In [ ]:
# Run ALNS algorithm
best_solution, best_charging_solution = alns.run()

print(f"Best solution objective value: {best_solution.objective_function():.2f}")
print(f"Improvement: {initial_solution.objective_function() - best_solution.objective_function():.2f}")
print(f"Number of vehicles used: {len(best_solution.routes)}")

## 10. Visualize Solutions

Finally, let's visualize the solutions to understand the routes and assignments:

In [ ]:
# Visualize initial solution
print("Initial Solution:")
initial_solution.plot_solution()

In [ ]:
# Visualize best solution found by ALNS
print("Best Solution (ALNS):")
best_solution.plot_solution()

## Conclusion

Congratulations! You've successfully:

1. ✅ Created a synthetic delivery map
2. ✅ Generated demand data for restaurant-customer pairs
3. ✅ Built a PDPTW problem instance
4. ✅ Generated an initial solution using greedy insertion
5. ✅ Improved the solution using ALNS algorithm
6. ✅ Visualized the routing solutions

### Next Steps

- Try modifying parameters (number of vehicles, battery capacity, etc.)
- Experiment with different removal/repair operators
- Use real-world map data with `RealDataMap`
- Check out the sensitivity analysis tutorial for more advanced scenarios

### Key Takeaways

- VRP Toolkit provides a modular architecture for VRP problems
- ALNS is effective for solving complex PDPTW problems
- The framework supports both synthetic and real-world data
- Visualization helps understand routing decisions